# CSV -> JSON

## Imports og tilkobling

In [8]:
import os
import pandas as pd
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import holidays

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

container_name = "trafoforbruksdata"

## Hente CSV fra blob

In [ ]:
blob_name = "BREIVE.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = blob_client.download_blob().readall()

df = pd.read_csv(io.BytesIO(data))

### Sjekk strukturen:

In [9]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 5837952 entries, 0 to 5837951
Data columns (total 5 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   MeteringPointAnonymous  str    
 1   TimestampUtc            str    
 2   Value                   float64
 3   TransformerStation      str    
 4   ConsumptionCode         int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 222.7 MB


,MeteringPointAnonymous,TimestampUtc,Value,TransformerStation,ConsumptionCode
0,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-17 23:00:00.0000000,5.63,BREIVE,36
1,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 00:00:00.0000000,4.21,BREIVE,36
2,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 01:00:00.0000000,3.34,BREIVE,36
3,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 02:00:00.0000000,3.82,BREIVE,36
4,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 03:00:00.0000000,3.04,BREIVE,36


## Lag nye kolonner

Omgjører tidskolonnen til datetime:

In [11]:
df["TimestampUtc"] = pd.to_datetime(df["TimestampUtc"])

### Helg (ja/nei)

0-4 = ukedag & 5-6 = helg

In [12]:
df["helg"] = df["TimestampUtc"].dt.weekday >= 5

### Helligdag (ja/nei)

In [13]:
no_holidays = holidays.Norway()

df["helligdag"] = df["TimestampUtc"].dt.date.isin(no_holidays)

### Dag/natt

dag = 06–22 & natt = 22–06

In [14]:
df["hour"] = df["TimestampUtc"].dt.hour

df["dag_natt"] = df["hour"].apply(
    lambda x: "dag" if 6 <= x < 22 else "natt"
)

In [17]:
df.head()

,MeteringPointAnonymous,TimestampUtc,Value,TransformerStation,ConsumptionCode,helg,helligdag,hour,dag_natt
0,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-17 23:00:00,5.63,BREIVE,36,False,False,23,natt
1,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 00:00:00,4.21,BREIVE,36,True,False,0,natt
2,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 01:00:00,3.34,BREIVE,36,True,False,1,natt
3,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 02:00:00,3.82,BREIVE,36,True,False,2,natt
4,fcf4855f-feca-467b-bad2-ff524b7fdca3,2025-01-18 03:00:00,3.04,BREIVE,36,True,False,3,natt


## Konvertere til JSON

In [18]:
json_data = df.to_json(orient="records", date_format="iso")

## Laste JSON til Blob

In [19]:
output_blob_name = "breive_forbruk.json"

blob_client_output = blob_service_client.get_blob_client(
    container=container_name,
    blob=output_blob_name
)

blob_client_output.upload_blob(json_data, overwrite=True)

print("JSON lastet opp til blob!")

JSON lastet opp til blob!
